# Práctica 5 - Aprendizaje No Supervisado I: Reducción de la Dimensionalidad

**Materia:** Introducción al Aprendizaje Automático  
**Tema:** Aplicacion de fundamentos matematicos, probabilidad y estadistica a traves de un piplenie de AA

**Nombre y apellido:**  
**Fecha de entrega:**  Miércoles 06/05/2026, 23:59



---

## Objetivos

En este cuadernillo vamos a estudiar qué ocurre cuando un dataset tiene muchas variables y queremos representarlo usando menos dimensiones.

Trabajaremos con **MNIST**: imágenes de dígitos escritos a mano.

Al finalizar, deberías poder explicar:

- qué significa reducir dimensionalidad;
- qué intenta preservar PCA;
- por qué t-SNE y UMAP son útiles para visualización;
- por qué una buena visualización no garantiza un mejor modelo.

## Parte 0 - Cargar librerías

Primero importamos las librerías necesarias.

> Puede ser necesario instalar `umap-learn`.

In [ ]:
# Descomentá esta línea si UMAP no está instalado:
# !pip install umap-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import make_swiss_roll
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from tensorflow.keras.datasets import mnist

import warnings
warnings.filterwarnings("ignore")

# Intentamos importar UMAP
try:
    import umap
    UMAP_DISPONIBLE = True
except ImportError:
    UMAP_DISPONIBLE = False
    print("UMAP no está instalado. En Colab ejecutá: !pip install umap-learn -q")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Parte 1 — Error por datos faltantes

Vamos a aplicar algunos métodos para lidiar con datos faltantes.

Corré el código de abajo. Vamos a generar datos faltantes sintéticos en los dígitos de MNIST.

In [ ]:

# Cargar datos
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalizar
X_train = X_train / 255.0
X_test = X_test / 255.0

# Aplanar imágenes (28x28 → 784) y no usamos todos los datos
X_train = X_train.reshape(-1, 784)[:5000]
X_test = X_test.reshape(-1, 784)[:5000]

# Introducir valoers faltantes (10%)
def add_missing(X, missing_rate=0.1):
    X_missing = X.copy()
    mask = np.random.rand(*X.shape) < missing_rate
    X_missing[mask] = np.nan
    return X_missing

X_train_missing = add_missing(X_train, 0.1)
X_test_missing = add_missing(X_test, 0.1)

print("Ejemplos con datos faltantes:", np.isnan(X_train_missing).mean())

Visualicemos algunas imágenes son píxeles faltantes.

In [ ]:
def show_image(x, title=""):
    plt.imshow(x.reshape(28,28), cmap="gray")
    plt.title(title)
    plt.axis("off")

idx = 0
plt.figure(figsize=(8,3))

plt.subplot(1,2,1)
show_image(X_train[idx], "Original")

plt.subplot(1,2,2)
show_image(X_train_missing[idx], "Con valores faltantes")

plt.show()

##1.1 Métodos simples (media / mediana)

Te doy el código, sólo correlo.



In [ ]:
from sklearn.impute import SimpleImputer

# Media
imputer_mean = SimpleImputer(strategy="mean")
X_train_mean = imputer_mean.fit_transform(X_train_missing)
X_test_mean = imputer_mean.transform(X_test_missing)

# Mediana
imputer_median = SimpleImputer(strategy="median")
X_train_median = imputer_median.fit_transform(X_train_missing)
X_test_median = imputer_median.transform(X_test_missing)

##1.2. kNN Imputation (basado en datos)


Te doy el código, sólo correlo. Tarda un poco.


In [ ]:
from sklearn.impute import KNNImputer

imputer_knn = KNNImputer(n_neighbors=5)

X_train_knn = imputer_knn.fit_transform(X_train_missing)

##1.3. Método avanzado (Iterative / tipo MICE)

Te doy el código, sólo correlo si tenés tiempo. Tarda un poco bastante (más de 15 minutos). Podés achicar el número de muestras. Sino, saltealo.


In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

imputer_iter = IterativeImputer(max_iter=3, random_state=42)

X_train_iter = imputer_iter.fit_transform(X_train_missing)

## Visualización

In [ ]:
nums = [1,5,10,20]

for i in nums:
    plt.figure(figsize=(18,3))

    plt.subplot(1,6,1)
    show_image(X_train[i], "Original")

    plt.subplot(1,6,2)
    show_image(X_train_missing[i], "Faltantes")

    plt.subplot(1,6,3)
    show_image(X_train_mean[i], "Media")

    plt.subplot(1,6,4)
    show_image(X_train_median[i], "Mediana")

    plt.subplot(1,6,5)
    show_image(X_train_knn[i], "kNN")

    #descomentá estas lineas si corriste MICE
    #plt.subplot(1,6,6)
    #show_image(X_train_iter[i], "MICE / Iterative")

    plt.show()

## Pregunta 1.1
Compará las imágenes imputadas con media/mediana vs kNN.

¿Qué tipo de estructura (bordes, trazos del dígito) se pierde en cada caso?
Relacioná esto con:


*   métodos simples introducen sesgo
*   métodos basados en datos usan estructuras locales


_Reemplazá este texto por tu respuesta._

## Pregunta 1.2

Dado todo lo observado:

¿Existe un método “mejor” de imputación? Considerá tiempo, inspeccioná imágenes, pensá en que este dataset va a ser usado para clasificación.

_Reemplazá este texto por tu respuesta._

# Parte 2 - MNIST como datos de alta dimensión

Recordemos: MNIST contiene imágenes de dígitos escritos a mano, del 0 al 9.

Cada imagen es un vector:

$
x \in \mathbb{R}^{784}
$

Es decir, cada imagen vive en un espacio $\mathbb{R}^{784}$

In [ ]:
#Cargamos devuelta por las dudas
from tensorflow.keras.datasets import mnist

(X_train_img, y_train), (X_test_img, y_test) = mnist.load_data()

print("Forma original de X_train:", X_train_img.shape)
print("Forma original de X_test:", X_test_img.shape)

#No nos interesan las etiquetas
print("Forma de y_train:", y_train.shape)

In [ ]:
X_train = X_train_img.reshape(len(X_train_img), -1)
X_test = X_test_img.reshape(len(X_test_img), -1)

print("Forma vectorizada de X_train:", X_train.shape)
print("Forma vectorizada de X_test:", X_test.shape)

### Submuestreo para visualización

t-SNE y UMAP pueden ser costosos si usamos todos los datos.  
Para este cuadernillo vamos a tomar una muestra más pequeña.

Esto no cambia la idea conceptual: seguimos trabajando con puntos en $\mathbb{R}^{784}$.

In [ ]:
# Tomamos una muestra para acelerar las visualizaciones
n_muestras = 3000

indices = np.random.choice(len(X_train), size=n_muestras, replace=False)

X = X_train[indices]
y = y_train[indices]

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

### Escalado de datos

PCA, t-SNE y UMAP se basan en distancias o varianzas.  
Por eso conviene escalar los datos.

En imágenes como MNIST, los píxeles ya están en una escala común: valores entre 0 y 255.  
Igualmente, para PCA y métodos basados en distancia, vamos a estandarizar.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Media aproximada:", np.mean(X_scaled))
print("Desvío estándar aproximado:", np.std(X_scaled))

## 2.1. PCA en MNIST

PCA busca nuevas direcciones, llamadas componentes principales, que preserven la mayor varianza posible.

Recordemos:

- PC1: dirección de máxima varianza.
- PC2: dirección ortogonal a PC1 con máxima varianza restante.
- PCA es un método lineal.

Fijate que pca_2d hace todos los cálculos que vimos en teoría por atrás, sin que tengamos que hacer nada más que darle las imágenes.

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)

print("Forma reducida:", X_pca_2d.shape)
print("Varianza explicada por PC1 y PC2:", pca_2d.explained_variance_ratio_)
print("Varianza total explicada por 2 componentes:", pca_2d.explained_variance_ratio_.sum())

### Pregunta conceptual

¿Cuántos componentes principales podríamos tener en MNIST en vez de 2?

_Reemplazá este texto por tu respuesta._


In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], s=8, cmap="tab10", alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("MNIST reducido a 2D con PCA")
plt.show()

### Pregunta 2.1.1


¿Ves alguna información a simple vista que te permita saber que esto representa muestras de distintos dígitos?


_Reemplazá este texto por tu respuesta._

Vamos a visualizar lo anterior de PCA pero ahora vamos a colorear por etiquetas (las tenemos en este caso y sólo las vamos a usar para ver si PCA captura algo de similitud entre el dígito y la etiqueta.)

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, s=8, cmap="tab10", alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("MNIST reducido a 2D con PCA")
plt.colorbar(scatter, label="Dígito")
plt.show()

### Pregunta 2.1.2

Observá el gráfico de PCA coloreado.

¿Los dígitos quedan perfectamente separados?


_Reemplazá este texto por tu respuesta._

### Pregunta 2.1.3

¿Podemos concluir que esas clases son inseparables en el espacio original? ¿Por qué?

_Reemplazá este texto por tu respuesta._

### Varianza explicada acumulada

Una pregunta importante en PCA es:

> ¿Cuántas componentes necesito para conservar una proporción razonable de la información?

Vamos a calcular la varianza explicada acumulada.

Hacemos PCA pero con todos los componentes.

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

var_acumulada = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(var_acumulada)
plt.xlabel("Número de componentes")
plt.ylabel("Varianza explicada acumulada")
plt.title("Varianza explicada acumulada en MNIST")
plt.grid(True)
plt.show()

for umbral in [0.50, 0.70, 0.80, 0.90, 0.95]:
    n_comp = np.argmax(var_acumulada >= umbral) + 1
    print(f"Componentes necesarias para explicar {int(umbral*100)}% de la varianza: {n_comp}")

### Pregunta 2.1.4

Dados los resultados anteriores, si usamos solo 2 componentes, ¿estamos conservando mucha o poca varianza?

Entonces, ¿por qué igual puede ser útil PCA en 2D?

_Reemplazá este texto por tu respuesta._

## 2.2. t-SNE en MNIST

t-SNE es un método no lineal usado principalmente para visualización.

A diferencia de PCA, t-SNE no busca preservar varianza global.  
Busca preservar vecindades locales:

$
\text{puntos cercanos en alta dimensión} \rightarrow \text{puntos cercanos en baja dimensión}
$

Podés variar `perplexity` pero no te lo recomiendo para este trabajo.

Va a tardar un poco.


In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=RANDOM_STATE
)

X_tsne_2d = tsne.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_tsne_2d[:, 0], X_tsne_2d[:, 1], s=8, cmap="tab10", alpha=0.7)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("MNIST reducido a 2D con t-SNE")
plt.show()

### Pregunta 2.2.1


¿Ves alguna información a simple vista que te permita saber que esto representa muestras de distintos dígitos?

_Reemplazá este texto por tu respuesta._

In [ ]:
#nos fijamos el mismo gráfico pero coloreado por etiquetas
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_tsne_2d[:, 0], X_tsne_2d[:, 1], c=y, s=8, cmap="tab10", alpha=0.7)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("MNIST reducido a 2D con t-SNE")
plt.colorbar(scatter, label="Dígito")
plt.show()

## Pregunta 2.2.2

Hacé una descripción breve de lo que observás en el gráfico anterior. Asocialo con el objetivo de t-SNE. ¿Se podría usar esta 'clusterización' en 2D para un clasificador?

_Reemplazá este texto por tu respuesta._

## 2.3. UMAP en MNIST

UMAP también es un método no lineal de reducción de dimensionalidad.  
Se usa mucho para visualización y exploración de datos.

Intuitivamente:

- construye un grafo de vecinos en alta dimensión;
- intenta preservar esa estructura en baja dimensión;
- suele ser más rápido que t-SNE;
- puede preservar algo más de estructura global, aunque también puede distorsionar.


In [ ]:
if UMAP_DISPONIBLE:
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        random_state=RANDOM_STATE
    )

    X_umap_2d = reducer.fit_transform(X_scaled)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_umap_2d[:, 0], X_umap_2d[:, 1],  s=8, cmap="tab10", alpha=0.7)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title("MNIST reducido a 2D con UMAP")
    plt.show()
else:
    print("UMAP no está disponible. Ejecutá: !pip install umap-learn -q y reiniciá el entorno.")

In [ ]:
    #Ploteamos por color
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_umap_2d[:, 0], X_umap_2d[:, 1], c=y, s=8, cmap="tab10", alpha=0.7)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title("MNIST reducido a 2D con UMAP")
    plt.colorbar(scatter, label="Dígito")
    plt.show()

### Ejercicio
Variá los hiperpámetros `n_neighbors=15`, y `min_dist=0.1`, dos de cada uno e imprimí la figura, coloreada por etiquetas.

In [ ]:
#Tu código aquì

### Pregunta 2.3.1

Hacé un resumen de lo que observás visualmente de UMAP ¿Hubieron cambios significativos al cambiar los hiperparámetros?

_Reemplazá este texto por tu respuesta._

## 2.4. Comparación lado a lado

Vamos a graficar los tres métodos juntos para compararlos visualmente.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, s=6, cmap="tab10", alpha=0.7)
axes[0].set_title("PCA")
axes[0].set_xlabel("Componente 1")
axes[0].set_ylabel("Componente 2")

axes[1].scatter(X_tsne_2d[:, 0], X_tsne_2d[:, 1], c=y, s=6, cmap="tab10", alpha=0.7)
axes[1].set_title("t-SNE")
axes[1].set_xlabel("Dimensión 1")
axes[1].set_ylabel("Dimensión 2")

if UMAP_DISPONIBLE:
    axes[2].scatter(X_umap_2d[:, 0], X_umap_2d[:, 1], c=y, s=6, cmap="tab10", alpha=0.7)
    axes[2].set_title("UMAP")
else:
    axes[2].text(0.5, 0.5, "UMAP no disponible", ha="center", va="center")
    axes[2].set_title("UMAP")

axes[2].set_xlabel("Dimensión 1")
axes[2].set_ylabel("Dimensión 2")

plt.tight_layout()
plt.show()

## Pregunta 2.4.1

Hacé una comparación de los tres métodos. Repasamos la teoría:

*   Método lineal/no lineal?
*   Tiempo de cómputo?
*   Clusterización marcada?

Cuál de estos métodos usarías para disminuir dimensión antes de darlo a un clasificador?


_Reemplazá este texto por tu respuesta._

## 2.5. ¿Reducir dimensión mejora un modelo?

Hasta ahora usamos reducción de dimensionalidad para visualizar.  
Pero también podemos preguntarnos si reducir dimensión ayuda a entrenar un modelo supervisado.

Vamos a comparar una regresión logística usando:

1. píxeles originales;
2. componentes principales de PCA;

Para que sea rápido, usaremos una muestra.

In [ ]:
# Usamos un subconjunto más grande pero manejable
n_modelo = 10000
idx_modelo = np.random.choice(len(X_train), size=n_modelo, replace=False)

X_modelo = X_train[idx_modelo]
y_modelo = y_train[idx_modelo]

X_tr, X_val, y_tr, y_val = train_test_split(
    X_modelo, y_modelo, test_size=0.25, random_state=RANDOM_STATE, stratify=y_modelo
)

scaler_modelo = StandardScaler()
X_tr_scaled = scaler_modelo.fit_transform(X_tr)
X_val_scaled = scaler_modelo.transform(X_val)

# Modelo con datos originales
clf_original = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
clf_original.fit(X_tr_scaled, y_tr)

pred_original = clf_original.predict(X_val_scaled)
acc_original = accuracy_score(y_val, pred_original)

print("Accuracy usando 784 variables originales:", acc_original)

Cambiá `n_componentes_modelo` por la cantidad que dio que explica la mayor varianza.

In [ ]:
# Modelo con PCA
n_componentes_modelo = ...

pca_modelo = PCA(n_components=n_componentes_modelo, random_state=RANDOM_STATE)

X_tr_pca = pca_modelo.fit_transform(X_tr_scaled)
X_val_pca = pca_modelo.transform(X_val_scaled)

clf_pca = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
clf_pca.fit(X_tr_pca, y_tr)

pred_pca = clf_pca.predict(X_val_pca)
acc_pca = accuracy_score(y_val, pred_pca)

print(f"Accuracy usando {n_componentes_modelo} componentes de PCA:", acc_pca)
print("Varianza explicada acumulada:", pca_modelo.explained_variance_ratio_.sum())

### Pregunta 2.5.1

¿El modelo con PCA tuvo mejor, peor o similar desempeño?
¿Reducir dimensionalidad garantiza mejorar el desempeño?


_Reemplazá este texto por tu respuesta._


### Pregunta 2.5.2 (Sólo posgrado)

¿Por qué PCA puede actuar como una forma indirecta de regularización?



_Reemplazá este texto por tu respuesta._

### Ejercicio 2.5

Entrená una regresión logística con UMAP. Calculá la exactitud. No se puede usar t-SNE para una clasificación, como ya vimos. Elegí el número de componentes que quieras.

In [ ]:
import umap

umap_model = umap.UMAP(n_components=..., random_state=RANDOM_STATE)

X_tr_umap = umap_model.fit_transform(X_tr_scaled)
X_val_umap = umap_model.transform(X_val_scaled)

clf_umap = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
clf_umap.fit(X_tr_umap, y_tr)

pred_umap = clf_umap.predict(X_val_umap)
acc_umap = accuracy_score(y_val, pred_umap)

print(f"Accuracy para UMAP:", acc_umap)

### Conclusión

Escribí una breve conclusión comparando las tres exactitudes en la clasificación: usando todas las features, PCA, UMAP.

_Reemplazá este texto por tu respuesta._

# Entrega

Antes de entregar, verificá lo siguiente:

- [ ] completé todas las consignas;
- [ ] ejecuté el cuaderno de principio a fin;
- [ ] dejé los resultados visibles;
- [ ] guardé el archivo con el nombre correcto.

Fin de la práctica.